# 03 – Evaluate Cellpose-SAM against the gold truth

**Purpose:** Segment the gold-truth frames of sequence 01 with the pretrained Cellpose-SAM model
and measure how well it agrees with the manual reference. Silver truth is evaluated the same way
as a baseline for what "good" means on this dataset.

**Input:** `data/raw/Fluo-N2DL-HeLa/01`, `01_GT/SEG`, `01_ST/SEG`
**Output:** `results/segmentation_metrics_cellpose_seq01.csv` (small table, tracked by git)

### Evaluation logic
1. **IoU** (intersection over union) for every pair of reference and predicted nucleus:
   overlap area / combined area. 1 = identical, 0 = no overlap.
2. **Match** = IoU > 0.5. Above 0.5 each object can have at most one partner
   (two partners would each need more than half of the union), so matching is unique without an
   assignment algorithm.
3. **Detection metrics – only on fully annotated frames (13, 52):**
   TP = matches, FP = unmatched predictions, FN = unmatched reference nuclei;
   precision = TP/(TP+FP), recall = TP/(TP+FN), F1 = harmonic mean of both.
4. **SEG-like score – on all 28 gold-truth frames:** mean IoU per *reference* nucleus (0 if not matched).
   It only looks at annotated nuclei, so it is valid on partially annotated frames.
   Note: the official CTC SEG measure uses a slightly different matching rule
   (prediction covers > 50 % of the reference pixels); values are close but not identical.

## 1. Imports and settings

In [1]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile
from skimage.segmentation import relabel_sequential

DATA = Path("../data/raw/Fluo-N2DL-HeLa")
RESULTS = Path("../results")
SEQ = "01"
FULL_FRAMES = [13, 52]   # gold-truth frames with (almost) all nuclei annotated (notebook 02)
IOU_THRESHOLD = 0.5

## 2. Load the gold-truth frames
Only frames with a gold-truth file are needed. For each: raw image, gold truth (GT), silver truth (ST).

In [2]:
def frame_number(path):
    return int("".join(c for c in path.stem if c.isdigit()))

gt_files = sorted((DATA / f"{SEQ}_GT" / "SEG").glob("man_seg*.tif"))
frames = [frame_number(f) for f in gt_files]

raw = {t: tifffile.imread(DATA / SEQ / f"t{t:03d}.tif") for t in frames}
gt  = {t: tifffile.imread(f) for t, f in zip(frames, gt_files)}
st  = {t: tifffile.imread(DATA / f"{SEQ}_ST" / "SEG" / f"man_seg{t:03d}.tif") for t in frames}

print("gold-truth frames:", frames)
print("fully annotated frames present:", [t for t in FULL_FRAMES if t in frames])

gold-truth frames: [12, 13, 14, 15, 20, 21, 22, 23, 25, 29, 38, 39, 40, 44, 50, 51, 52, 53, 54, 55, 62, 76, 77, 78, 79, 80, 81, 88]
fully annotated frames present: [13, 52]


## 3. IoU and matching functions
`relabel_sequential` renumbers labels to 1…n. Gold-truth label IDs are not consecutive, and the
IoU table uses label values as row/column positions.

The overlap table is built in one step with `np.bincount`: each pixel gets a combined code
`gt_label × (n_pred + 1) + pred_label`; counting the codes gives the number of pixels for every
(gt, pred) label pair – a contingency table, like a cross-tabulation in R.

In [3]:
def iou_matrix(gt_lab, pred_lab):
    """IoU for every (reference, prediction) pair. Rows = reference objects, columns = predictions."""
    n_gt, n_pred = int(gt_lab.max()), int(pred_lab.max())
    codes = gt_lab.ravel().astype(np.int64) * (n_pred + 1) + pred_lab.ravel().astype(np.int64)
    overlap = np.bincount(codes, minlength=(n_gt + 1) * (n_pred + 1)).reshape(n_gt + 1, n_pred + 1)
    area_gt = overlap.sum(axis=1, keepdims=True)     # pixels per reference object
    area_pred = overlap.sum(axis=0, keepdims=True)   # pixels per predicted object
    union = area_gt + area_pred - overlap
    iou = overlap / np.maximum(union, 1)
    return iou[1:, 1:]                               # drop background row and column


def evaluate(gt_lab, pred_lab, thresh=IOU_THRESHOLD):
    """Detection metrics and SEG-like score for one frame."""
    gt_lab = relabel_sequential(gt_lab)[0]
    pred_lab = relabel_sequential(pred_lab)[0]
    iou = iou_matrix(gt_lab, pred_lab)
    n_gt, n_pred = iou.shape

    hits = iou > thresh                       # unique matches (see text above)
    tp = int(hits.sum())
    fp, fn = n_pred - tp, n_gt - tp
    best_iou_per_gt = iou.max(axis=1) if n_pred else np.zeros(n_gt)

    precision = tp / n_pred if n_pred else np.nan
    recall = tp / n_gt if n_gt else np.nan
    return {
        "n_gt": n_gt, "n_pred": n_pred, "tp": tp, "fp": fp, "fn": fn,
        "precision": precision, "recall": recall,
        "f1": 2 * tp / (n_gt + n_pred) if (n_gt + n_pred) else np.nan,
        "mean_iou_matched": iou[hits].mean() if tp else np.nan,
        "seg_like": np.where(best_iou_per_gt > thresh, best_iou_per_gt, 0).mean() if n_gt else np.nan,
    }

### Sanity check of the functions
Before trusting a metric on real data, test it where the answer is known:
- an image compared with itself must give IoU 1 and no errors;
- removing one object must give exactly one false negative;
- a shifted copy must lower the IoU but keep the counts plausible.

In [4]:
toy = np.zeros((40, 40), dtype=np.uint16)
toy[2:12, 2:12] = 1
toy[20:30, 5:15] = 2
toy[15:25, 25:35] = 3

missing = toy.copy(); missing[missing == 3] = 0
shifted = np.roll(toy, 2, axis=1)

for name, pred in [("identical", toy), ("one missing", missing), ("shifted by 2 px", shifted)]:
    r = evaluate(toy, pred)
    print(f"{name:16s} tp={r['tp']} fp={r['fp']} fn={r['fn']}  "
          f"mean IoU={r['mean_iou_matched']:.2f}  seg_like={r['seg_like']:.2f}")

identical        tp=3 fp=0 fn=0  mean IoU=1.00  seg_like=1.00
one missing      tp=2 fp=0 fn=1  mean IoU=1.00  seg_like=0.67
shifted by 2 px  tp=3 fp=0 fn=0  mean IoU=0.67  seg_like=0.67


## 4. Segment with Cellpose-SAM
Same call as in notebook 01. Cellpose normalises intensities internally (percentile-based),
so the camera offset (~33,000) needs no correction. Runtime is recorded for the methods section.

In [6]:
from cellpose import models, core

print("GPU available:", core.use_gpu())
if "model" not in globals():
    model = models.CellposeModel(gpu=True)
if "pred" not in globals():
    pred = {}

# fully annotated frames first, then the rest
order = [t for t in FULL_FRAMES if t in frames] + [t for t in frames if t not in FULL_FRAMES]

t_start = time.time()
for i, t in enumerate(order, 1):
    if t in pred:                       # already done (e.g. before an interrupt)
        continue
    t0 = time.time()
    pred[t], _, _ = model.eval(raw[t])
    print(f"[{i}/{len(order)}] frame {t}: {pred[t].max()} nuclei, {time.time() - t0:.0f} s", flush=True)
print(f"total: {time.time() - t_start:.0f} s")

GPU available: True
[18/28] frame 53: 117 nuclei, 17 s
[19/28] frame 54: 116 nuclei, 17 s
[20/28] frame 55: 115 nuclei, 17 s
[21/28] frame 62: 130 nuclei, 17 s
[22/28] frame 76: 138 nuclei, 17 s
[23/28] frame 77: 138 nuclei, 18 s
[24/28] frame 78: 140 nuclei, 20 s
[25/28] frame 79: 141 nuclei, 22 s
[26/28] frame 80: 144 nuclei, 22 s
[27/28] frame 81: 145 nuclei, 22 s
[28/28] frame 88: 151 nuclei, 21 s
total: 209 s


## 5. Detection metrics on the fully annotated frames
Cellpose and silver truth side by side. Silver truth is itself an automatic segmentation
(fused from top challenge entries), so it is a realistic upper reference for a pretrained model.

In [7]:
rows = []
for t in FULL_FRAMES:
    if t not in frames:
        continue
    rows.append({"frame": t, "method": "cellpose-sam", **evaluate(gt[t], pred[t])})
    rows.append({"frame": t, "method": "silver truth", **evaluate(gt[t], st[t])})

full = pd.DataFrame(rows)
cols = ["frame", "method", "n_gt", "n_pred", "tp", "fp", "fn", "precision", "recall", "f1", "mean_iou_matched"]
print(full[cols].round(3).to_string(index=False))

 frame       method  n_gt  n_pred  tp  fp  fn  precision  recall    f1  mean_iou_matched
    13 cellpose-sam    59      58  54   4   5      0.931   0.915 0.923             0.843
    13 silver truth    59      59  59   0   0      1.000   1.000 1.000             0.920
    52 cellpose-sam   110     116 108   8   2      0.931   0.982 0.956             0.869
    52 silver truth   110     113 110   3   0      0.973   1.000 0.987             0.915


## 6. SEG-like score on all gold-truth frames
Valid on partially annotated frames because only annotated nuclei are scored.
`n_gt` shows how many nuclei each frame contributes.

In [8]:
rows = []
for t in frames:
    for method, lab in [("cellpose-sam", pred[t]), ("silver truth", st[t])]:
        r = evaluate(gt[t], lab)
        rows.append({"frame": t, "method": method, "n_gt": r["n_gt"], "seg_like": r["seg_like"],
                     "recall": r["recall"]})
seg = pd.DataFrame(rows)

wide = seg.pivot(index="frame", columns="method", values="seg_like")
print(wide.round(3).to_string())

# weighted by number of annotated nuclei, so that frames with 1-2 nuclei do not dominate
weighted = seg.groupby("method").apply(
    lambda d: np.average(d["seg_like"], weights=d["n_gt"]), include_groups=False)
print("\nSEG-like score, weighted by annotated nuclei:")
print(weighted.round(3).to_string())

method  cellpose-sam  silver truth
frame                             
12             0.723         0.799
13             0.772         0.920
14             0.595         0.763
15             0.774         0.871
20             0.759         0.885
21             0.828         0.802
22             0.624         0.812
23             0.742         0.848
25             0.812         0.891
29             0.799         0.871
38             0.749         0.902
39             0.697         0.891
40             0.661         0.826
44             0.692         0.889
50             0.902         0.850
51             0.289         0.787
52             0.853         0.915
53             0.390         0.742
54             0.704         0.890
55             0.740         0.875
62             0.801         0.890
76             0.754         0.881
77             0.809         0.852
78             0.865         0.843
79             0.784         0.850
80             0.775         0.858
81             0.864

## 7. Save the metrics
Small CSV tables go to `results/` and are tracked by git – they are the basis for figures and the README.

In [9]:
RESULTS.mkdir(exist_ok=True)
out = RESULTS / f"segmentation_metrics_cellpose_seq{SEQ}.csv"
pd.concat([full.assign(level="full_frame"), seg.assign(level="seg_all_frames")]).to_csv(out, index=False)
print("saved:", out)

saved: ../results/segmentation_metrics_cellpose_seq01.csv


## 8. Inspect errors in napari (frame 13)
Two extra layers mark the errors:
- **false positives:** predicted nuclei without a reference partner
- **false negatives:** reference nuclei that were not found

Look at *why* errors happen: touching nuclei, mitoses, dim nuclei, image border?

In [12]:
%gui qt
import napari

t = 51
g = relabel_sequential(gt[t])[0]
p = relabel_sequential(pred[t])[0]
iou = iou_matrix(g, p)
fp_ids = np.where(~(iou > IOU_THRESHOLD).any(axis=0))[0] + 1   # +1: matrix starts at label 1
fn_ids = np.where(~(iou > IOU_THRESHOLD).any(axis=1))[0] + 1

lo, hi = np.percentile(raw[t], [1, 99.8])
viewer = napari.Viewer()
viewer.add_image(raw[t], name=f"raw t{t}", colormap="gray", contrast_limits=(lo, hi))
viewer.add_labels(g, name="gold truth", opacity=0.4)
viewer.add_labels(p, name="cellpose-sam", opacity=0.4, visible=False)
if t in FULL_FRAMES:   # false positives are only meaningful when every nucleus is annotated
    viewer.add_labels(np.where(np.isin(p, fp_ids), p, 0), name="false positives")
viewer.add_labels(np.where(np.isin(g, fn_ids), g, 0), name="false negatives")
print(f"frame {t}: annotated {g.max()} | missed {len(fn_ids)}"
      + (f" | false positives {len(fp_ids)}" if t in FULL_FRAMES else " (partial frame: no FP count)"))
print("false positives:", len(fp_ids), "| false negatives:", len(fn_ids))

frame 51: annotated 3 | missed 2 (partial frame: no FP count)
false positives: 111 | false negatives: 2


## Results and observations (2026-10-30, sequence 01)

**Sanity check:** IoU/matching functions return the expected values on toy images
(identical → IoU 1; one object removed → 1 FN; 2-px shift of a 10×10 square → IoU 0.67).

**Runtime:** Cellpose-SAM on Apple M2 (MPS): ~17–22 s per 700 × 1100 frame (28 frames ≈ 8 min).

**Detection on fully annotated frames (pretrained Cellpose-SAM, no training on this dataset):**

| Frame | Method | GT | Pred | TP | FP | FN | Precision | Recall | F1 | IoU (matched) |
|---|---|---|---|---|---|---|---|---|---|---|
| 13 | Cellpose-SAM | 59 | 58 | 54 | 4 | 5 | 0.93 | 0.92 | 0.92 | 0.84 |
| 13 | Silver truth | 59 | 59 | 59 | 0 | 0 | 1.00 | 1.00 | 1.00 | 0.92 |
| 52 | Cellpose-SAM | 110 | 116 | 108 | 8 | 2 | 0.93 | 0.98 | 0.96 | 0.87 |
| 52 | Silver truth | 110 | 113 | 110 | 3 | 0 | 0.97 | 1.00 | 0.99 | 0.92 |

**SEG-like score (all 28 GT frames, weighted by annotated nuclei):** Cellpose-SAM 0.78, silver truth 0.88.
Lowest frames: 51 (0.29) and 53 (0.39) – only 3–4 annotated nuclei each, so one miss weighs heavily.

**Interpretation:**
- Cellpose finds nuclei almost as reliably as the silver truth, but its outlines agree less with the
  manual annotation (IoU 0.84–0.87 vs. 0.92) → room for improvement by training on this dataset.
- Silver truth is not an independent baseline: it is built from challenge methods that likely used
  annotations of this dataset; Cellpose has never seen it.
- Gold truth has very limited instance coverage by design (CTC). Only frames 13 and 52 are fully annotated;
  partial frames contain selected, often difficult nuclei.

**Error inspection (napari):**
- Frame 13 false positives: nuclei cut by the image border – most likely not annotated by convention,
  not true segmentation errors.
- Frame 13 false negatives: two very small signals; three normal-looking nuclei (cause not yet checked –
  possibly merged with or split from a neighbour).
- Frame 51: the three annotated nuclei are atypical – a small, line-like object (likely a metaphase plate),
  a bent nucleus with internal structure (prophase or apoptosis), and one normal-looking nucleus.
  → Cellpose-SAM struggles mainly with mitotic figures.

**Next:** quantify segmentation quality as a function of distance to division (lineage table);
compare with StarDist and with a model trained on this dataset.